# Single-class object of interest — Detectron2 v2 (BCE + Dice)

Same training recipe as `train_rtdetr_l_v2.ipynb`, with the smallest Detectron2-specific adapter so the two runs can be compared later.

| Setting | YOLO v2 | **This notebook** |
|---|---|---|
| mask loss | 0.5 BCE + 0.5 Dice | **same** |
| `cls` | 0.4 | **same** (`loss_cls` scaled by 0.4) |
| `mosaic` / `close_mosaic` | 0.4 / 10 | **same** |
| `cos_lr` | True | **same** (`WarmupCosineLR`) |
| `epochs` / `batch` / `imgsz` | 200 / 4 / 640 | **same** |
| `patience` | 20 | **same** (on val `segm/AP50`) |
| model | YOLO11l-seg | **Mask R-CNN R-50-FPN** (required) |

Same single-class `train_v2/dataset` (`nc: 1`, class `0` / `object`). Do **not** run the download from this notebook. Weights go to `train_v2/runs/detectron2/` — YOLO v1/v2 folders are not touched.

## 0. Download the dataset (run this in a terminal first)

Skip this if `train_v2/dataset` already exists from the YOLO runs.

From the **repo root** in PowerShell:

```powershell
python train_v2/utils/download_dataset.py
```

## 1. Setup and environment check

In [1]:
from pathlib import Path
import os
import sys

import torch
from detectron2.utils.logger import setup_logger

setup_logger()

HERE = Path.cwd().resolve()
if not (HERE / "utils" / "download_dataset.py").exists():
    HERE = HERE / "train_v2"

sys.path.insert(0, str(HERE / "utils"))
from detectron2_recipe import (
    RECIPE,
    ObjectTrainer,
    apply_bce_dice_mask_loss,
    build_cfg,
    print_recipe,
    register_splits,
)
from yolo_to_coco import convert_dataset

DATA_YAML = HERE / "dataset" / "data.yaml"
DATASET = HERE / "dataset"
RUNS = HERE / "runs"
PROJECT = RUNS / "detectron2"
RUN_NAME = "mask_rcnn_r50_object_bce_dice"
OUTPUT_DIR = PROJECT / RUN_NAME
YOLO_V2_DIR = RUNS / "v2" / "yolo11l_seg_object_bce_dice"

os.environ["MLFLOW_EXPERIMENT_NAME"] = "train_v2_detectron2_bce_dice"
os.environ["MLFLOW_RUN"] = RUN_NAME
os.environ["MLFLOW_TRACKING_URI"] = (RUNS / "mlflow").resolve().as_uri()

if OUTPUT_DIR.resolve() == YOLO_V2_DIR.resolve():
    raise RuntimeError("Detectron2 save path collided with the YOLO v2 run.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("TRAIN_V2 — DETECTRON2 SINGLE-CLASS SEGMENTATION (v2 BCE+Dice)")
print("=" * 70)
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print()
print(f"train_v2    : {HERE}")
print(f"data.yaml   : {DATA_YAML}")
print(f"yaml exists : {DATA_YAML.exists()}")
print(f"YOLO v2 dir : {YOLO_V2_DIR}  exists={YOLO_V2_DIR.exists()}")
print(f"d2 project  : {PROJECT}")
print(f"d2 run name : {RUN_NAME}")
print(f"d2 save dir : {OUTPUT_DIR}")
print(f"MLflow exp  : {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print(f"MLflow run  : {os.environ['MLFLOW_RUN']}")
if not DATA_YAML.exists():
    raise FileNotFoundError(
        "Dataset not found. From the repo root run:\n"
        "  python train_v2/utils/download_dataset.py"
    )
print("=" * 70)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


TRAIN_V2 — DETECTRON2 SINGLE-CLASS SEGMENTATION (v2 BCE+Dice)
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU

train_v2    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2
data.yaml   : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml
yaml exists : True
YOLO v2 dir : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v2\yolo11l_seg_object_bce_dice  exists=True
d2 project  : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\detectron2
d2 run name : mask_rcnn_r50_object_bce_dice
d2 save dir : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\detectron2\mask_rcnn_r50_object_bce_dice
MLflow exp  : train_v2_detectron2_bce_dice
MLflow run  : mask_rcnn_r50_object_bce_dice


## 2. Confirm the local set is single-class

Every polygon should already be class `0` after the download script. This cell only checks; it does not rewrite labels.

In [2]:
import yaml

cfg_yaml = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("nc   :", cfg_yaml.get("nc"))
print("names:", cfg_yaml.get("names"))

class_ids = set()
n_objects = 0
label_files = list((HERE / "dataset" / "labels").rglob("*.txt"))
for path in label_files:
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        class_ids.add(int(float(parts[0])))
        n_objects += 1

print(f"label files : {len(label_files)}")
print(f"objects     : {n_objects}")
print(f"class ids   : {sorted(class_ids)}")
if class_ids != {0}:
    raise ValueError(f"Expected only class 0, found {sorted(class_ids)}")
print("OK — single class `object` (id 0). SKU labels are ignored.")

nc   : 1
names: {0: 'object'}
label files : 2726
objects     : 6067
class ids   : [0]
OK — single class `object` (id 0). SKU labels are ignored.


## 3. YOLO-seg → COCO JSON

Detectron2 needs COCO instance JSON. Polygons stay the same; every object is `category_id=1` / `object`. Writes `train_v2/dataset/coco/{train,val,test}.json` and does not change the YOLO labels.

In [3]:
from detectron2.data import DatasetCatalog

coco_paths = convert_dataset(DATASET)
print(coco_paths)

names = register_splits(DATASET)
train_dicts = DatasetCatalog.get(names["train"])
print(f"registered : {names}")
print(f"train images (COCO): {len(train_dicts)}")

COCO train: 100%|██████████| 1909/1909 [00:02<00:00, 802.34file/s]


train: 1909 images, 4355 objects → C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\coco\train.json


COCO val: 100%|██████████| 403/403 [00:00<00:00, 809.74file/s]


val: 403 images, 811 objects → C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\coco\val.json


COCO test: 100%|██████████| 414/414 [00:00<00:00, 597.16file/s]


test: 414 images, 901 objects → C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\coco\test.json
{'train': WindowsPath('C:/Users/Haqkiem/OneDrive/UNIKL/July-2026/Competition/AIIC/PETROSAINS/train_v2/dataset/coco/train.json'), 'val': WindowsPath('C:/Users/Haqkiem/OneDrive/UNIKL/July-2026/Competition/AIIC/PETROSAINS/train_v2/dataset/coco/val.json'), 'test': WindowsPath('C:/Users/Haqkiem/OneDrive/UNIKL/July-2026/Competition/AIIC/PETROSAINS/train_v2/dataset/coco/test.json')}
[09/13 08:38:19 d2.data.datasets.coco]: Loaded 1909 images in COCO format from C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\coco\train.json
registered : {'train': 'petrosains_object_train', 'val': 'petrosains_object_val', 'test': 'petrosains_object_test'}
train images (COCO): 1909


## 4. Switch mask loss to BCE + Dice

Patches Detectron2 `mask_rcnn_loss` to **0.5 BCE + 0.5 Dice** (same hybrid as the YOLO v2 notebook). Box regression stays Detectron2 default (no DFL).

In [4]:
apply_bce_dice_mask_loss()

Detectron2 mask loss: 0.5 BCE + 0.5 Dice


## 5. Augmentation + solver (label-aware + light mosaic)

Keeps the v2 photometric / small-geometry augs, mosaic `0.4`, and disables mosaic for the last 10 epochs (`close_mosaic=10`). Cosine LR. `batch=4` and `imgsz=640` are unchanged.

In [5]:
cfg, iters_per_epoch, max_iter = build_cfg(
    OUTPUT_DIR, names["train"], names["val"], n_train=len(train_dicts)
)
print_recipe(cfg, iters_per_epoch, max_iter)

v2 recipe (kept) vs Detectron2 mapping:
  epochs         200
  batch          4
  imgsz          640
  cls            0.4
  mosaic         0.4
  close_mosaic   10
  cos_lr         True
  patience       20
  mask_loss      0.5 BCE + 0.5 Dice
  fliplr         0.5
  flipud         0.0
  degrees        8.0
  translate      0.05
  scale          0.1
  hsv_h          0.015
  hsv_s          0.5
  hsv_v          0.4
  mixup          0.0
  copy_paste     0.0
  perspective    0.0
  shear          0.0

  iters/epoch   477
  max_iter      95400  (= 477 x 200)
  close_mosaic  last 10 epochs → iter 90630
  optimizer     SGD  lr=0.005  cosine→5e-05
  AMP           True  (torch.amp.autocast('cuda'))
  workers       4
  output        C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\detectron2\mask_rcnn_r50_object_bce_dice


## 6. Train Mask R-CNN R-50-FPN (BCE + Dice, 200 epochs)

Weights land in `train_v2/runs/detectron2/mask_rcnn_r50_object_bce_dice/` (not the YOLO folders).

`batch=4` and `imgsz=640` are unchanged. Mosaic + Mask R-CNN can OOM on 6 GB; drop `BATCH` to `2` in `detectron2_recipe.py` if needed.

Best checkpoint is `model_best.pth` (highest val `segm/AP50`). Detectron2 reports AP on a **0–100** scale; divide by 100 when comparing to YOLO mAP.

In [ ]:
import mlflow

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

with mlflow.start_run(run_name=os.environ["MLFLOW_RUN"]):
    mlflow.log_params({k: str(v) for k, v in RECIPE.items()})
    mlflow.log_params(
        {
            "model": "mask_rcnn_R_50_FPN_3x",
            "iters_per_epoch": iters_per_epoch,
            "max_iter": max_iter,
            "optimizer": "SGD",
            "base_lr": cfg.SOLVER.BASE_LR,
        }
    )
    trainer = ObjectTrainer(cfg, train_dicts, iters_per_epoch)
    trainer.resume_or_load(resume=False)
    results = trainer.train()
    print(results)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 328, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 422, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\mlflow\store\tracking\file_store.py", line 1368, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-pa

[09/13 08:38:23 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

roi_heads.box_predictor.bbox_pred.{bias, weight}
roi_heads.box_predictor.cls_score.{bias, weight}
roi_heads.mask_head.predictor.{bias, weight}


[09/13 08:38:23 detectron2]: Starting training from iteration 0


c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4383.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0913 08:38:42.960835 13700 site-packages\torch\fx\_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


[09/13 08:39:08 d2.utils.events]:  epoch: 1/200  eta: 1 day, 8:31:42  iter: 19  total_loss: 1.678  loss_cls: 0.2344  loss_box_reg: 0.6269  loss_mask: 0.6039  loss_rpn_cls: 0.08989  loss_rpn_loc: 0.02387    time: 1.2681  last_time: 1.2344  data_time: 1.4036  last_data_time: 0.3710   lr: 7.1284e-05  max_mem: 1434M
[09/13 08:39:33 d2.utils.events]:  epoch: 1/200  eta: 1 day, 4:41:35  iter: 39  total_loss: 1.594  loss_cls: 0.2046  loss_box_reg: 0.594  loss_mask: 0.5405  loss_rpn_cls: 0.1378  loss_rpn_loc: 0.03697    time: 1.2710  last_time: 0.3931  data_time: 0.5778  last_data_time: 0.0041   lr: 0.00014106  max_mem: 1440M
[09/13 08:40:01 d2.utils.events]:  epoch: 1/200  eta: 1 day, 7:30:37  iter: 59  total_loss: 1.315  loss_cls: 0.1664  loss_box_reg: 0.5713  loss_mask: 0.4457  loss_rpn_cls: 0.08255  loss_rpn_loc: 0.02593    time: 1.3083  last_time: 0.3750  data_time: 0.8015  last_data_time: 0.0036   lr: 0.00021083  max_mem: 1440M
[09/13 08:40:30 d2.utils.events]:  epoch: 1/200  eta: 1 day,

## 7. Quick sanity check on a val image

Loads `model_best.pth` from this Detectron2 run. `conf=0.25` matches the YOLO v2 preview.

In [ ]:
from detectron2.engine import DefaultPredictor
from detectron2.utils.visualizer import ColorMode, Visualizer
from PIL import Image
import numpy as np

best = OUTPUT_DIR / "model_best.pth"
if not best.exists():
    raise FileNotFoundError(f"No weights yet: {best}")

val_images = sorted(
    p for p in (HERE / "dataset" / "images" / "val").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
sample = val_images[0]
print("Sample:", sample)

pred_cfg = cfg.clone()
pred_cfg.defrost()
pred_cfg.MODEL.WEIGHTS = str(best)
pred_cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.25
predictor = DefaultPredictor(pred_cfg)

im = np.array(Image.open(sample).convert("RGB"))[:, :, ::-1]  # RGB → BGR
out = predictor(im)
from detectron2.data import MetadataCatalog

vis = Visualizer(
    im[:, :, ::-1],
    metadata=MetadataCatalog.get(names["val"]),
    scale=1.0,
    instance_mode=ColorMode.IMAGE,
)
drawn = vis.draw_instance_predictions(out["instances"].to("cpu")).get_image()
preview = OUTPUT_DIR / "preview_val.jpg"
Image.fromarray(drawn).save(preview)
n = len(out["instances"])
print(f"detections: {n}  → {preview}")

## Maintained vs changed

### Maintained (fair-compare recipe)

- Same `train_v2/dataset`, single class `object`
- Mask loss **0.5 BCE + 0.5 Dice**
- `cls=0.4`
- mosaic `0.4`, `close_mosaic=10`
- cosine LR
- **200 epochs**, **batch=4**, **imgsz=640**
- Label-aware augs: flip 0.5, degrees ±8, translate 0.05, scale 0.10, HSV; no mixup / copy-paste / perspective / shear
- `patience=20` on val mask AP50
- Isolated save dir + MLflow experiment (does not overwrite YOLO v1/v2)

### Changed (Detectron2 adapter)

- **Model:** Mask R-CNN R-50-FPN (COCO 3x zoo), not YOLO11l-seg. Ultralytics RT-DETR-L has no mask head.
- **Labels:** extra COCO JSON (`dataset/coco/*.json`). YOLO txt files are unchanged.
- **Schedule unit:** `max_iter = (n_train // 4) * 200` instead of Ultralytics `epochs=`
- **Optimizer:** Detectron2 default **SGD** (scaled 0.005 @ batch 4). YOLO v2 used AdamW `lr0=0.01`
- **Box loss:** Smooth L1 / Detectron2 box loss. **No DFL**
- **Mosaic:** custom 2×2 canvas (Detectron2 has no Ultralytics mosaic)
- **Workers:** `NUM_WORKERS=4` with a shared mosaic flag so `close_mosaic` still works on Windows
- **ROI heads:** `BATCH_SIZE_PER_IMAGE=128` (default 512) for 6 GB VRAM
- **Metrics:** COCO AP is 0–100; YOLO mAP is 0–1. Compare `segm/AP50 / 100` to YOLO mask mAP50
- **Best weights:** `model_best.pth` on `segm/AP50` (YOLO `best.pt` uses mask+box mAP50-95 fitness)